In [2]:
from transformers import pipeline

mask_filler = pipeline("fill-mask")

input_sentence = "Hanoi is the <mask> of Vietnam."
predictions = mask_filler(input_sentence, top_k=5)

print(f"Câu gốc: {input_sentence}")
for pred in predictions:
    print(f"Dự đoán: '{pred['token_str']}' với độ tin cậy: {pred['score']:.4f}")
    print(f" -> Câu hoàn chỉnh: {pred['sequence']}")


No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8 (https://huggingface.co/distilbert/distilroberta-base).
Using a pipeline without specifying a model name and revision in production is not recommended.
Some weights of the model checkpoint at distilbert/distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Câu gốc: Hanoi is the <mask> of Vietnam.
Dự đoán: ' capital' với độ tin cậy: 0.9341
 -> Câu hoàn chỉnh: Hanoi is the capital of Vietnam.
Dự đoán: ' Republic' với độ tin cậy: 0.0300
 -> Câu hoàn chỉnh: Hanoi is the Republic of Vietnam.
Dự đoán: ' Capital' với độ tin cậy: 0.0105
 -> Câu hoàn chỉnh: Hanoi is the Capital of Vietnam.
Dự đoán: ' birthplace' với độ tin cậy: 0.0054
 -> Câu hoàn chỉnh: Hanoi is the birthplace of Vietnam.
Dự đoán: ' heart' với độ tin cậy: 0.0014
 -> Câu hoàn chỉnh: Hanoi is the heart of Vietnam.


### Bài 1: Masked Language Modeling
1. Mô hình đã dự đoán đúng từ capital không?

Có.
Các mô hình BERT được huấn luyện trên corpora lớn như Wikipedia, nơi câu “Hanoi is the capital of Vietnam” xuất hiện nhiều lần, nên mô hình gần như chắc chắn dự đoán đúng token capital với độ tin cậy cao.

2. Tại sao các mô hình Encoder-only như BERT phù hợp cho tác vụ này?

BERT phù hợp cho Masked Language Modeling vì:

Bidirectional attention: BERT nhìn được cả trái và phải của token <mask>, nên hiểu đầy đủ ngữ cảnh để dự đoán từ bị che.

Được huấn luyện trực tiếp bằng MLM: Trong quá trình pretrain, BERT che 15% token => mô hình tối ưu hoàn toàn cho nhiệm vụ điền từ.

Không dùng cơ chế sinh chuỗi: Khác GPT, BERT không cần sinh tiếp token => tập trung tối đa vào việc hiểu ngữ cảnh.

In [3]:
from transformers import pipeline

generator = pipeline("text-generation")
prompt = "The best thing about learning NLP is"

generated_texts = generator(prompt, max_length=50, num_return_sequences=1)

print(f"Câu mồi: '{prompt}'")
for text in generated_texts:
    print("Văn bản được sinh ra:")
    print(text['generated_text'])


No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Câu mồi: 'The best thing about learning NLP is'
Văn bản được sinh ra:
The best thing about learning NLP is that it's a very simple, flexible way to learn. It's a wonderful way to learn and an interesting way to grow.

1. Learn To Write

Learning how to write is one of the most important aspects of being a writer. You can learn how to write as many lines of code as you want by reading books, writing articles, and reading articles on different types of computer terminals or writing HTML.

2. Learn To Read

Learning to read is one of the most important aspects of being a writer. You can learn how to read, write, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read, read,

### Bài 2: Next Token Prediction
1. Kết quả sinh ra có hợp lý không?

Có.
Các mô hình GPT sinh văn bản dựa trên xác suất token tiếp theo, nên kết quả thường mạch lạc, ngữ nghĩa ổn và liên quan trực tiếp đến chủ đề “learning NLP”.

2. Tại sao mô hình Decoder-only như GPT phù hợp cho tác vụ này?

GPT phù hợp cho Next Token Prediction vì:

Unidirectional (left-to-right): GPT chỉ nhìn token phía trước => mô phỏng chính xác quá trình sinh văn bản.

Mục tiêu huấn luyện là Next Token Prediction: Mỗi bước dự đoán token tiếp theo => hoàn toàn phù hợp cho text generation.

Kiến trúc đơn giản và nhanh hơn: Không có encoder => sinh chữ nhanh, mượt.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModel

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

sentences = ["This is a sample sentence."]

inputs = tokenizer(sentences, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state
attention_mask = inputs['attention_mask']

mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
sum_embeddings = torch.sum(last_hidden_state * mask_expanded, 1)
sum_mask = torch.clamp(mask_expanded.sum(1), min=1e-9)

sentence_embedding = sum_embeddings / sum_mask

print("Vector biểu diễn của câu:")
print(sentence_embedding)
print("Kích thước:", sentence_embedding.shape)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Vector biểu diễn của câu:
tensor([[-6.3874e-02, -4.2837e-01, -6.6779e-02, -3.8430e-01, -6.5784e-02,
         -2.1826e-01,  4.7636e-01,  4.8659e-01,  4.0647e-05, -7.4273e-02,
         -7.4740e-02, -4.7635e-01, -1.9773e-01,  2.4824e-01, -1.2162e-01,
          1.6678e-01,  2.1045e-01, -1.4576e-01,  1.2636e-01,  1.8635e-02,
          2.4640e-01,  5.7090e-01, -4.7014e-01,  1.3782e-01,  7.3650e-01,
         -3.3808e-01, -5.0331e-02, -1.6452e-01, -4.3517e-01, -1.2900e-01,
          1.6516e-01,  3.4004e-01, -1.4930e-01,  2.2422e-02, -1.0488e-01,
         -5.1916e-01,  3.2964e-01, -2.2162e-01, -3.4206e-01,  1.1993e-01,
         -7.0148e-01, -2.3126e-01,  1.1224e-01,  1.2550e-01, -2.5191e-01,
         -4.6374e-01, -2.7261e-02, -2.8415e-01, -9.9249e-02, -3.7017e-02,
         -8.9192e-01,  2.5005e-01,  1.5816e-01,  2.2701e-01, -2.8497e-01,
          4.5300e-01,  5.0945e-03, -7.9441e-01, -3.1008e-01, -1.7403e-01,
          4.3029e-01,  1.6816e-01,  1.0590e-01, -4.8987e-01,  3.1856e-01,
          3.

### Bài 3: Sentence Embedding bằng Mean Pooling
1. Kích thước (chiều) của vector biểu diễn là bao nhiêu? Nó tương ứng với tham số nào của BERT?

Vector có kích thước 768 chiều.

Đây là giá trị của tham số hidden_size = 768 trong mô hình bert-base-uncased.

2. Tại sao cần sử dụng attention_mask khi thực hiện Mean Pooling?

Vì:

attention_mask giúp loại bỏ token đệm (padding) khỏi việc tính trung bình.

Nếu không dùng mask, các vector [PAD] (thường là zero-vector) sẽ kéo trung bình về 0 => làm sai lệch embedding thực.

Masking đảm bảo chỉ tính trung bình các token thật của câu.